In [0]:
df_logs = spark.read.csv(
    "/Volumes/workspace/default/dataset_streamings_databricks/logs_streaming.csv",
    header=True,
    inferSchema=True
)

display(df_logs)

In [0]:
#visualizar todas as vezes em que o país está nulo
from pyspark.sql.functions import col

display(df_logs.filter(
    col("country").isNull()
))

In [0]:
#Contabilizando ocorrências de country nulo
from pyspark.sql.functions import count, when

df_logs.select(
    count(
        when(col("country").isNull(), True)
    ).alias("total_country_nulo")
).display()

In [0]:
#tratando o campo nulo com Unknown
df_logs = df_logs.fillna(
    {"country": "Unknown"}
)

display(df_logs)

In [0]:
#Tratando o campo nulo removendo coluna
df_logs_sem_nulos = df_logs.dropna(
    subset=["user_id"]
)

display(df_logs_sem_nulos)

**Tratamento de tipos e formatos:**

In [0]:
df_logs.printSchema()

**Tratando coluna numérica com try_cast e replace**

Utilizamos tryCast (na API SQL, via expr como try_cast), que tenta converter todos os valores sem interromper o processamento. Se houver algum valor inconsistente ou inválido, a execução não é interrompida e segue adiante. Por isso, tryCast é preferível a cast

In [0]:
from pyspark.sql.functions import expr

In [0]:
df_logs = df_logs.withColumn(
    "watch_time_minutes",
    expr(
        "try_cast("
            "replace(watch_time_minutes, 'err', '0')"
        "as double)"
    )
)

In [0]:
df_logs.printSchema()

**Identificando formatos de data inconsistentes**


In [0]:
df_logs.select("watch_date").display()

**Convertendo datas com to_date e validando**

In [0]:
from pyspark.sql.functions import to_date

df_logs = df_logs.withColumn(
    "watch_date",
    to_date("watch_date", 'yyyy-MM-dd')
)

df_logs.printSchema()

In [0]:
df_logs.select(
    "watch_date",
    "watch_time_minutes"
).display()

**Tratando coluna numérica com try_cast e replace:**

In [0]:
from pyspark.sql.functions import expr

df_logs = df_logs.withColumn(
    "watch_time_minutes",
    expr(
        "try_cast("
            "replace(watch_time_minutes, 'err', '0')"
        "as double)"
    )
)

In [0]:
df_logs.printSchema()

In [0]:
df_logs.select("watch_date").display()

**Tentando try_to_date e lidando com cache:**

In [0]:
from pyspark.sql.functions import try_to_date

In [0]:
df_logs = df_logs.withColumn(
    "watch_date",
    try_to_date("watch_date", 'yyyy-MM-dd')
)

In [0]:
df_logs.select("watch_date").display()

**Joins e enriquecimento de dados:**

In [0]:
df_catalogo = spark.read.csv(
    "/Volumes/workspace/default/dataset_streamings_databricks/catalogo_filmes.csv",
    header=True,
    inferSchema=False
)

df_usuario = spark.read.csv(
    "/Volumes/workspace/default/dataset_streamings_databricks/usuarios.csv",
    header=True,
    inferSchema=False
)

In [0]:
display(df_logs)

In [0]:
display(df_catalogo)

In [0]:

display(df_usuario)

**Realizando a primeira junção e investigando ausências:**

In [0]:
df_enriquecido = df_logs.join(
    df_catalogo,
    on="movie_id",
    how = "left"
)

display(df_enriquecido)

In [0]:
df_enriquecido.filter(
    col("title").isNull()
).display()

**Combinando dados de usuários e consolidando a visão:**

In [0]:
df_final = df_enriquecido.join(
    df_usuarios,
    on="user_id",
    how="left"
)

display(df_final)